In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
import click
import tkinter as tk
from tkinter import filedialog
import locan as lc


# =============================================================================
# STEP 1 — Data loading
# =============================================================================
class LocData:
    def __init__(self, data):
        self.data = data
        self.coordinate_keys = ['position_x', 'position_y']

    @staticmethod
    def from_dataframe(df):
        return LocData(df)


def load_data(file_type, paths):
    locdata_list = []
    for path in tqdm(paths, desc="Loading data files"):
        if file_type == "ELYRA":
            locdata = lc.load_Elyra_file(path=path)
        elif file_type == "THUNDERSTORM":
            locdata = lc.load_thunderstorm_file(path=path)
        else:
            raise ValueError("Unsupported file type. Supported types are 'ELYRA' and 'THUNDERSTORM'.")

        if not hasattr(locdata, 'coordinate_keys') or not locdata.coordinate_keys:
            locdata.coordinate_keys = ['position_x', 'position_y']

        if 'frame' not in locdata.data.columns:
            raise ValueError(f"'frame' column not found in file: {path}")

        locdata_list.append((path, locdata))
    return locdata_list


def select_files_for_condition(condition_name):
    print(f"\nSelect files for condition: {condition_name}")
    root = tk.Tk()
    root.withdraw()
    file_paths = filedialog.askopenfilenames(
        title=f"Select localization data files for {condition_name}"
    )
    file_type = input("Enter the file type (ELYRA or THUNDERSTORM): ").strip().upper()
    return file_type, file_paths


def get_psf_column(df, file_type):
    if file_type == "ELYRA":
        if 'psf_half_width' not in df.columns:
            raise KeyError("Expected column 'psf_half_width' not found in ELYRA data.")
        return 'psf_half_width'
    elif file_type == "THUNDERSTORM":
        if 'psf_sigma' in df.columns:
            return 'psf_sigma'
        elif 'psf_half_width' in df.columns:
            return 'psf_half_width'
        else:
            raise KeyError("Neither 'psf_sigma' nor 'psf_half_width' exist in the THUNDERSTORM data.")
    else:
        raise ValueError("Unsupported file type.")


def get_filtering_thresholds(locdata_with_types):
    pooled_uncertainty = []
    pooled_psf = []
    pooled_intensity = []
    pooled_frame = []

    for file_type, locdata in locdata_with_types:
        df = locdata.data
        psf_col = get_psf_column(df, file_type)
        required_columns = ['uncertainty', 'intensity', 'frame', psf_col]
        for col in required_columns:
            if col not in df.columns:
                raise KeyError(f"The specified column '{col}' does not exist in the data.")
        pooled_uncertainty.append(pd.to_numeric(df['uncertainty'], errors='coerce'))
        pooled_psf.append(pd.to_numeric(df[psf_col], errors='coerce'))
        pooled_intensity.append(pd.to_numeric(df['intensity'], errors='coerce'))
        pooled_frame.append(pd.to_numeric(df['frame'], errors='coerce'))

    pooled_uncertainty = pd.concat(pooled_uncertainty, ignore_index=True)
    pooled_uncertainty = pooled_uncertainty[np.isfinite(pooled_uncertainty)]
    pooled_psf = pd.concat(pooled_psf, ignore_index=True)
    pooled_psf = pooled_psf[np.isfinite(pooled_psf)]
    pooled_intensity = pd.concat(pooled_intensity, ignore_index=True)
    pooled_intensity = pooled_intensity[np.isfinite(pooled_intensity)]
    pooled_frame = pd.concat(pooled_frame, ignore_index=True)
    pooled_frame = pooled_frame[np.isfinite(pooled_frame)]

    loc_precision_max_default = pooled_uncertainty.quantile(0.95)
    psf_max_default = pooled_psf.quantile(0.95)
    intensity_min_default = pooled_intensity.quantile(0.05)
    intensity_max_default = pooled_intensity.quantile(0.95)
    frame_min_default = pooled_frame.quantile(0.05)

    print("\nDefault percentile values for filtering parameters (calculated from pooled data):")
    print(f"Localization precision max (95th percentile): {loc_precision_max_default}")
    print(f"PSF max (95th percentile): {psf_max_default}")
    print(f"Intensity min (5th percentile): {intensity_min_default}")
    print(f"Intensity max (95th percentile): {intensity_max_default}")
    print(f"Frame min (5th percentile): {frame_min_default}")

    thresholds = {
        'loc_precision_max': click.prompt("Enter localization precision max", default=float(loc_precision_max_default), type=float),
        'psf_max': click.prompt("Enter PSF max", default=float(psf_max_default), type=float),
        'intensity_min': click.prompt("Enter intensity min", default=float(intensity_min_default), type=float),
        'intensity_max': click.prompt("Enter intensity max", default=float(intensity_max_default), type=float),
        'frame_min': click.prompt("Enter frame min", default=float(frame_min_default), type=float),
    }
    return thresholds


def filter_data(locdata, file_type, thresholds):
    initial_count = len(locdata.data)
    psf_col = get_psf_column(locdata.data, file_type)
    required_columns = ['uncertainty', 'intensity', 'frame', psf_col]
    for col in required_columns:
        if col not in locdata.data.columns:
            raise KeyError(f"The specified column '{col}' does not exist in the data.")

    filtered_data = locdata.data[
        (locdata.data['uncertainty'] <= thresholds['loc_precision_max']) &
        (locdata.data[psf_col] <= thresholds['psf_max']) &
        (locdata.data['intensity'] >= thresholds['intensity_min']) &
        (locdata.data['intensity'] <= thresholds['intensity_max']) &
        (locdata.data['frame'] >= thresholds['frame_min'])
    ]

    final_count = len(filtered_data)
    percentage_removed = 100 * (initial_count - final_count) / initial_count
    print(f"Initial localizations: {initial_count}")
    print(f"Final localizations: {final_count}")
    print(f"Percentage removed: {percentage_removed:.2f}%")
    return LocData.from_dataframe(filtered_data)


# =============================================================================
# Main workflow — load files for all 3 conditions: PFA, BP, Cryo
# =============================================================================
if __name__ == "__main__":
    import sys

    if 'ipykernel' in sys.modules:
        from IPython import get_ipython
        get_ipython().run_line_magic('matplotlib', 'inline')

    # -------------------------------------------------------------------------
    # Load data for condition 1: PFA
    # -------------------------------------------------------------------------
    condition1_name = "PFA"
    file_type1, file_paths1 = select_files_for_condition(condition1_name)
    locdata_list1 = load_data(file_type1, file_paths1)

    # -------------------------------------------------------------------------
    # Load data for condition 2: BP
    # -------------------------------------------------------------------------
    condition2_name = "BP"
    file_type2, file_paths2 = select_files_for_condition(condition2_name)
    locdata_list2 = load_data(file_type2, file_paths2)

    # -------------------------------------------------------------------------
    # Load data for condition 3: Cryo
    # -------------------------------------------------------------------------
    condition3_name = "Cryo"
    file_type3, file_paths3 = select_files_for_condition(condition3_name)
    locdata_list3 = load_data(file_type3, file_paths3)

    # Store all three conditions together
    locdata_combined = [
        (condition1_name, file_type1, locdata_list1),
        (condition2_name, file_type2, locdata_list2),
        (condition3_name, file_type3, locdata_list3)
    ]

    num_cpus = os.cpu_count()
    print(f"\nNumber of available CPU cores: {num_cpus}")

    # Pool all data for shared threshold estimation
    pooled_locdata_with_types = []
    for _, file_type, locdata_list in locdata_combined:
        for _, locdata in locdata_list:
            pooled_locdata_with_types.append((file_type, locdata))

    print("\nEnter the filtering thresholds to be applied to all three conditions:")
    thresholds = get_filtering_thresholds(pooled_locdata_with_types)

    # Filter all conditions
    all_filtered_locdata = {}
    for condition_name, file_type, locdata_list in locdata_combined:
        print(f"\nProcessing condition: {condition_name}")
        filtered_locdata_list = []
        for file_name, locdata in tqdm(locdata_list, desc=f"Filtering data for {condition_name}"):
            filtered_locdata = filter_data(locdata, file_type, thresholds)
            filtered_locdata_list.append((file_name, filtered_locdata))
        all_filtered_locdata[condition_name] = filtered_locdata_list

    print("\nFiltering complete for all three conditions.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# =============================================================================
# STEP 2 — Manual visual QC of filtered localization heatmaps
# =============================================================================

condition_order = ['PFA', 'BP', 'Cryo']
cmap = 'hot'
bins = 200

approved_locdata = {condition: [] for condition in condition_order}
qc_summary = []

for condition_name in condition_order:
    filtered_locdata_list = all_filtered_locdata.get(condition_name, [])
    print(f"\nInspecting condition: {condition_name}")
    print("-" * 60)

    for file_idx, (file_name, filtered_locdata) in enumerate(filtered_locdata_list, start=1):
        data = filtered_locdata.data

        if data.empty:
            print(f"\nFile {file_idx}: {file_name}")
            print("This dataset is empty after filtering and will be marked as rejected.")
            qc_summary.append({
                "condition": condition_name,
                "file_name": file_name,
                "n_localizations": 0,
                "decision": "rejected_empty"
            })
            continue

        x = data['position_x']
        y = data['position_y']
        intensities = data['intensity']

        heatmap, xedges, yedges = np.histogram2d(
            x, y, bins=bins, weights=intensities, density=False
        )
        log_heatmap = np.log1p(heatmap)

        fig, ax = plt.subplots(figsize=(6, 6))
        im = ax.imshow(
            log_heatmap.T,
            extent=[x.min(), x.max(), y.min(), y.max()],
            origin='lower',
            cmap=cmap,
            aspect='equal'
        )
        ax.set_title(f"{condition_name} | File {file_idx}\n{file_name}", fontsize=11)
        ax.set_xlabel("Position X")
        ax.set_ylabel("Position Y")
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Log(Summed Intensity)")
        plt.show()

        while True:
            decision = input("Keep this file for downstream analysis? (y/n): ").strip().lower()
            if decision in ["y", "yes"]:
                approved_locdata[condition_name].append((file_name, filtered_locdata))
                qc_summary.append({
                    "condition": condition_name,
                    "file_name": file_name,
                    "n_localizations": len(data),
                    "decision": "approved"
                })
                print("Approved.\n")
                break
            elif decision in ["n", "no"]:
                qc_summary.append({
                    "condition": condition_name,
                    "file_name": file_name,
                    "n_localizations": len(data),
                    "decision": "rejected"
                })
                print("Rejected.\n")
                break
            else:
                print("Invalid input. Please enter 'y' or 'n'.")

qc_summary_df = pd.DataFrame(qc_summary)
print("\nManual QC complete.")
print("\nQC summary:")
print(qc_summary_df)
print("\nApproved files per condition:")
for condition_name, files in approved_locdata.items():
    print(f"{condition_name}: {len(files)} approved file(s)")

In [ ]:
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.stats import mannwhitneyu
import seaborn as sns


# =============================================================================
# STEP 3A — Density-based square ROI picker (3 conditions: PFA, BP, Cryo)
# =============================================================================

# Create output directory on Desktop
date_tag = datetime.now().strftime("%Y%m%d")
analysis_output_dir = Path.home() / "Desktop" / f"{date_tag}_glyco_analysis"
analysis_output_dir.mkdir(parents=True, exist_ok=True)
print(f"Analysis output directory:\n{analysis_output_dir}")


# -----------------------------------------------------------------------------
# Color palette for 3 conditions
# -----------------------------------------------------------------------------
PFA_COLOR   = "#481E52"   # purple
BP_COLOR    = "#A7C946"   # yellow-green
CRYO_COLOR  = "#4A979F"   # teal

PALETTE = {"PFA": PFA_COLOR, "BP": BP_COLOR, "Cryo": CRYO_COLOR}
CONDITION_ORDER = ["PFA", "BP", "Cryo"]

sns.set_style("white")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def prepare_condition_dataframe_list(approved_locdata, condition_name):
    dataset_list = []
    for file_name, loc in approved_locdata.get(condition_name, []):
        df = loc.data.copy().rename(columns={"position_x": "x", "position_y": "y"})
        dataset_list.append((file_name, df))
    return dataset_list


def get_square_bounds(df):
    xmin, xmax = df["x"].min(), df["x"].max()
    ymin, ymax = df["y"].min(), df["y"].max()
    x_center = 0.5 * (xmin + xmax)
    y_center = 0.5 * (ymin + ymax)
    side = max(xmax - xmin, ymax - ymin)
    return {
        "xmin": x_center - side / 2,
        "xmax": x_center + side / 2,
        "ymin": y_center - side / 2,
        "ymax": y_center + side / 2,
        "side": side
    }


def generate_tiled_roi_candidates(df, region_size=3000, min_points=20):
    bounds = get_square_bounds(df)
    x_starts = np.arange(bounds["xmin"], bounds["xmax"] - region_size + 1e-9, region_size)
    y_starts = np.arange(bounds["ymin"], bounds["ymax"] - region_size + 1e-9, region_size)
    candidates = []
    for x0 in x_starts:
        for y0 in y_starts:
            x1, y1 = x0 + region_size, y0 + region_size
            in_tile = (df["x"] >= x0) & (df["x"] < x1) & (df["y"] >= y0) & (df["y"] < y1)
            n_points = int(in_tile.sum())
            if n_points >= min_points:
                candidates.append({"xmin": x0, "xmax": x1, "ymin": y0, "ymax": y1,
                                    "n_points": n_points, "density": n_points / (region_size ** 2)})
    if not candidates:
        return pd.DataFrame(columns=["xmin", "xmax", "ymin", "ymax", "n_points", "density"])
    return pd.DataFrame(candidates).sort_values(by=["density", "n_points"], ascending=False).reset_index(drop=True)


def extract_roi_points(df, region):
    return df[
        (df["x"] >= region["xmin"]) & (df["x"] < region["xmax"]) &
        (df["y"] >= region["ymin"]) & (df["y"] < region["ymax"])
    ].copy()


def plot_candidate_roi(df, region, condition_name, dataset_idx, file_name, roi_idx):
    bounds = get_square_bounds(df)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(df["x"], df["y"], s=1, alpha=0.35, color="black")
    rect = patches.Rectangle(
        (region["xmin"], region["ymin"]),
        region["xmax"] - region["xmin"],
        region["ymax"] - region["ymin"],
        linewidth=2, edgecolor="red", facecolor="none"
    )
    ax.add_patch(rect)
    ax.set_xlim(bounds["xmin"], bounds["xmax"])
    ax.set_ylim(bounds["ymin"], bounds["ymax"])
    ax.set_aspect("equal")
    ax.set_title(
        f"{condition_name} | Dataset {dataset_idx + 1} | ROI candidate {roi_idx + 1}\n{Path(file_name).name}",
        fontsize=11
    )
    ax.set_xlabel("Position X")
    ax.set_ylabel("Position Y")
    return fig, ax


def select_density_ranked_rois_for_condition(dataset_list, condition_name, region_size=3000, n_rois=5, min_points=20):
    selected_regions_for_condition = []
    for dataset_idx, (file_name, df) in enumerate(dataset_list):
        print(f"\nSelecting ROIs for {condition_name} dataset {dataset_idx + 1}")
        print(f"File: {file_name}")
        candidates_df = generate_tiled_roi_candidates(df=df, region_size=region_size, min_points=min_points)
        if candidates_df.empty:
            print("No valid ROI candidates found for this dataset.")
            selected_regions_for_condition.append({"file_name": file_name, "regions": []})
            continue
        selected_regions = []
        candidate_pointer = 0
        while len(selected_regions) < n_rois and candidate_pointer < len(candidates_df):
            region = candidates_df.iloc[candidate_pointer].to_dict()
            fig, ax = plot_candidate_roi(df=df, region=region, condition_name=condition_name,
                                         dataset_idx=dataset_idx, file_name=file_name, roi_idx=len(selected_regions))
            plt.show()
            print(f"Candidate stats | n_points = {int(region['n_points'])}, density = {region['density']:.4e}")
            decision = input("Accept this ROI? (y = yes / n = no / q = stop this dataset): ").strip().lower()
            plt.close(fig)
            if decision in ["y", "yes"]:
                selected_regions.append(region)
                print(f"ROI {len(selected_regions)} accepted.")
            elif decision in ["n", "no"]:
                print("ROI rejected. Showing next candidate.")
            elif decision in ["q", "quit", "stop"]:
                print("Stopping ROI selection for this dataset.")
                break
            else:
                print("Invalid input. Showing next candidate anyway.")
            candidate_pointer += 1
        selected_regions_for_condition.append({"file_name": file_name, "regions": selected_regions})
    return selected_regions_for_condition


def p_to_stars(p):
    if pd.isna(p): return "NaN"
    elif p < 0.0001: return "****"
    elif p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"


def compare_two_conditions_mannwhitney(df, metric_col, cond_a, cond_b):
    vals_a = pd.to_numeric(df.loc[df["condition"] == cond_a, metric_col], errors="coerce").dropna()
    vals_b = pd.to_numeric(df.loc[df["condition"] == cond_b, metric_col], errors="coerce").dropna()
    if len(vals_a) == 0 or len(vals_b) == 0:
        return np.nan, np.nan
    stat, p_value = mannwhitneyu(vals_a, vals_b, alternative="two-sided")
    return stat, p_value


def add_significance_annotation_3groups(ax, df, metric_col, y_padding_fraction=0.08):
    """
    Add pairwise significance brackets for all 3 condition pairs:
    PFA vs BP, PFA vs Cryo, BP vs Cryo
    """
    pairs = [("PFA", "BP", 0, 1), ("BP", "Cryo", 1, 2), ("PFA", "Cryo", 0, 2)]
    vals = pd.to_numeric(df[metric_col], errors="coerce").dropna()
    if len(vals) == 0:
        return
    y_max = vals.max()
    y_range = vals.max() - vals.min() if vals.max() > vals.min() else 1
    offsets = [0.10, 0.22, 0.34]  # stacked brackets at different heights
    for (cond_a, cond_b, x_a, x_b), offset in zip(pairs, offsets):
        _, p_value = compare_two_conditions_mannwhitney(df, metric_col, cond_a, cond_b)
        y_line = y_max + offset * y_range
        y_text = y_line + 0.02 * y_range
        ax.plot([x_a, x_a, x_b, x_b],
                [y_line, y_line + 0.015 * y_range, y_line + 0.015 * y_range, y_line],
                lw=1.2, c="black")
        ax.text((x_a + x_b) / 2, y_text, p_to_stars(p_value),
                ha="center", va="bottom", fontsize=12, fontweight="bold")
    current_bottom, current_top = ax.get_ylim()
    ax.set_ylim(current_bottom, max(current_top, y_max + 0.45 * y_range))


def style_condition_ticklabels(ax):
    for tick_label in ax.get_xticklabels():
        tick_label.set_fontweight("bold")
        tick_label.set_fontsize(12)


def add_violin_median_lines(ax, df, metric_col, line_width=2.2, half_width=0.16):
    for i, condition in enumerate(CONDITION_ORDER):
        vals = pd.to_numeric(df.loc[df["condition"] == condition, metric_col], errors="coerce").dropna()
        if len(vals) == 0:
            continue
        median_val = np.median(vals)
        ax.plot([i - half_width, i + half_width], [median_val, median_val],
                color="white", linewidth=line_width, solid_capstyle="round", zorder=5)
        ax.plot([i - half_width, i + half_width], [median_val, median_val],
                color="black", linewidth=0.8, solid_capstyle="round", zorder=6)


# -----------------------------------------------------------------------------
# Prepare dataset lists for all 3 conditions
# -----------------------------------------------------------------------------
pfa_dataset_list  = prepare_condition_dataframe_list(approved_locdata, "PFA")
bp_dataset_list   = prepare_condition_dataframe_list(approved_locdata, "BP")
cryo_dataset_list = prepare_condition_dataframe_list(approved_locdata, "Cryo")


# -----------------------------------------------------------------------------
# Run ROI selection for all 3 conditions
# -----------------------------------------------------------------------------
selected_regions_pfa  = select_density_ranked_rois_for_condition(pfa_dataset_list,  "PFA",  region_size=3000, n_rois=5, min_points=20)
selected_regions_bp   = select_density_ranked_rois_for_condition(bp_dataset_list,   "BP",   region_size=3000, n_rois=5, min_points=20)
selected_regions_cryo = select_density_ranked_rois_for_condition(cryo_dataset_list, "Cryo", region_size=3000, n_rois=5, min_points=20)

selected_regions_by_condition = {
    "PFA":  selected_regions_pfa,
    "BP":   selected_regions_bp,
    "Cryo": selected_regions_cryo
}


# -----------------------------------------------------------------------------
# Build and save ROI summary table
# -----------------------------------------------------------------------------
roi_summary_rows = []
for condition_name, dataset_regions in selected_regions_by_condition.items():
    for dataset_idx, dataset_entry in enumerate(dataset_regions):
        file_name = dataset_entry["file_name"]
        for roi_idx, region in enumerate(dataset_entry["regions"], start=1):
            roi_summary_rows.append({
                "condition": condition_name,
                "dataset_index": dataset_idx + 1,
                "file_name": file_name,
                "roi_index": roi_idx,
                "xmin": region["xmin"], "xmax": region["xmax"],
                "ymin": region["ymin"], "ymax": region["ymax"],
                "n_points": int(region["n_points"]),
                "density": region["density"]
            })

roi_summary_df = pd.DataFrame(roi_summary_rows)

if not roi_summary_df.empty:
    roi_medians = roi_summary_df.groupby("condition")["n_points"].median().to_dict()
else:
    roi_medians = {c: np.nan for c in CONDITION_ORDER}

roi_summary_df["condition_median_n_points"] = roi_summary_df["condition"].map(roi_medians)

roi_summary_path = analysis_output_dir / "roi_selection_summary.csv"
roi_summary_df.to_csv(roi_summary_path, index=False)
print(f"\nROI selection complete.\nROI summary saved to:\n{roi_summary_path}")


# -----------------------------------------------------------------------------
# Plot localizations per ROI for all 3 conditions
# -----------------------------------------------------------------------------
if not roi_summary_df.empty:
    plot_df = roi_summary_df.copy()
    plot_df = plot_df[plot_df["condition"].isin(CONDITION_ORDER)].copy()
    plot_df["n_points"] = pd.to_numeric(plot_df["n_points"], errors="coerce")
    plot_df = plot_df.dropna(subset=["n_points"])

    fig, ax = plt.subplots(figsize=(6.5, 6))
    sns.violinplot(data=plot_df, x="condition", y="n_points", hue="condition",
                   order=CONDITION_ORDER, hue_order=CONDITION_ORDER, palette=PALETTE,
                   inner=None, cut=0, linewidth=1.2, dodge=False, legend=False, ax=ax)
    add_violin_median_lines(ax, plot_df, "n_points")
    sns.swarmplot(data=plot_df, x="condition", y="n_points", order=CONDITION_ORDER,
                  color="black", size=3, alpha=0.8, ax=ax)
    ax.set_title("Localizations per ROI", fontsize=16)
    ax.set_xlabel("")
    ax.set_ylabel("Number of localizations per ROI", fontsize=14)
    style_condition_ticklabels(ax)
    add_significance_annotation_3groups(ax, plot_df, "n_points")

    roi_count_plot_path = analysis_output_dir / "roi_localizations_per_roi_PFA_BP_Cryo.png"
    fig.savefig(roi_count_plot_path, dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"ROI localization comparison figure saved to:\n{roi_count_plot_path}")
    print("\nMedian number of localizations per ROI:")
    for cond in CONDITION_ORDER:
        print(f"{cond}: {roi_medians.get(cond, np.nan)}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import distance
from scipy import stats


# =============================================================================
# STEP 3B — Ripley's H analysis on selected ROIs (3 conditions)
# =============================================================================

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

INDIVIDUAL_CURVE_COLOR = "0.75"


def manual_ripley_k(df_roi, max_radius, num_radii=50, max_points=5000):
    points = df_roi[["x", "y"]].values
    num_points = len(points)

    if num_points < 2:
        radii = np.linspace(0, max_radius, num_radii)
        return radii, np.full_like(radii, np.nan), np.full_like(radii, np.nan)

    # Wenn zu viele Punkte: zufällig auf max_points reduzieren
    if num_points > max_points:
        idx = np.random.choice(num_points, max_points, replace=False)
        points = points[idx]
        num_points = max_points

    area = (df_roi["x"].max() - df_roi["x"].min()) * (df_roi["y"].max() - df_roi["y"].min())
    radii = np.linspace(0, max_radius, num_radii)
    K_values = np.zeros(num_radii)
    pairwise_distances = distance.cdist(points, points)

    for i, radius in enumerate(radii):
        within_radius = np.sum(pairwise_distances < radius) - num_points
        K_values[i] = (within_radius / num_points) * (area / num_points)

    H_values = np.sqrt(np.maximum(K_values / np.pi, 0)) - radii
    return radii, K_values, H_values

def collect_ripley_h_from_selected_rois(dataset_list, selected_regions_for_condition, max_radius=500, num_radii=50):
    h_curves = []
    metadata_rows = []
    radii = np.linspace(0, max_radius, num_radii)
    dataset_lookup = {file_name: df for file_name, df in dataset_list}
    for dataset_idx, dataset_entry in enumerate(selected_regions_for_condition, start=1):
        file_name = dataset_entry["file_name"]
        regions = dataset_entry["regions"]
        df = dataset_lookup[file_name]
        for roi_idx, region in enumerate(regions, start=1):
            df_roi = extract_roi_points(df, region)
            if len(df_roi) < 2:
                continue
            radii, _, h_values = manual_ripley_k(df_roi=df_roi, max_radius=max_radius, num_radii=num_radii)
            if np.all(np.isnan(h_values)):
                continue
            h_curves.append(h_values)
            metadata_rows.append({"dataset_index": dataset_idx, "file_name": file_name,
                                   "roi_index": roi_idx, "n_points": len(df_roi)})
    return radii, h_curves, metadata_rows


def get_curve_peak(radii, h_values):
    peak_idx = int(np.nanargmax(h_values))
    return radii[peak_idx], h_values[peak_idx]


def plot_ripley_mean_sem_single_condition(radii, h_curve_list, condition_name, color, save_dir):
    if len(h_curve_list) == 0:
        print(f"No valid Ripley H curves found for {condition_name}. Skipping.")
        return
    h_array = np.vstack(h_curve_list)
    mean_h = np.nanmean(h_array, axis=0)
    sem_h = stats.sem(h_array, axis=0, nan_policy="omit")
    peak_r, peak_h = get_curve_peak(radii, mean_h)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for h_curve in h_curve_list:
        ax.plot(radii, h_curve, color=INDIVIDUAL_CURVE_COLOR, linewidth=1.0, alpha=0.8)
    ax.plot(radii, mean_h, color=color, linewidth=2.5,
            label=f"Mean H(r) | Peak H = {peak_h:.2f}, r = {peak_r:.1f}")
    ax.fill_between(radii, mean_h - sem_h, mean_h + sem_h, color=color, alpha=0.20, label="SEM")
    ax.axhline(0, color="black", linestyle="--", linewidth=1.2)
    ax.scatter([peak_r], [peak_h], s=40, color=color, zorder=5)
    ax.set_title(f"Ripley's H(r) — {condition_name} (Mean ± SEM)")
    ax.set_xlabel("Radius r")
    ax.set_ylabel("Ripley's H(r)")
    ax.legend(frameon=False, loc="best")
    save_path = Path(save_dir) / f"ripley_H_{condition_name}_mean_SEM.png"
    fig.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved:\n{save_path}")


def plot_ripley_median_iqr_single_condition(radii, h_curve_list, condition_name, color, save_dir):
    if len(h_curve_list) == 0:
        print(f"No valid Ripley H curves found for {condition_name}. Skipping.")
        return
    h_array = np.vstack(h_curve_list)
    median_h = np.nanmedian(h_array, axis=0)
    q25 = np.nanpercentile(h_array, 25, axis=0)
    q75 = np.nanpercentile(h_array, 75, axis=0)
    peak_r, peak_h = get_curve_peak(radii, median_h)
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    for h_curve in h_curve_list:
        ax.plot(radii, h_curve, color=INDIVIDUAL_CURVE_COLOR, linewidth=1.0, alpha=0.8)
    ax.plot(radii, median_h, color=color, linewidth=2.5,
            label=f"Median H(r) | Peak H = {peak_h:.2f}, r = {peak_r:.1f}")
    ax.fill_between(radii, q25, q75, color=color, alpha=0.20, label="IQR")
    ax.axhline(0, color="black", linestyle="--", linewidth=1.2, label="CSR: H(r) = 0")
    ax.scatter([peak_r], [peak_h], s=40, color=color, zorder=5)
    ax.set_title(f"Ripley's H(r) — {condition_name} (Median + IQR)")
    ax.set_xlabel("Radius r")
    ax.set_ylabel("Ripley's H(r)")
    ax.legend(frameon=False, loc="best")
    save_path = Path(save_dir) / f"ripley_H_{condition_name}_median_IQR.png"
    fig.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved:\n{save_path}")


# -----------------------------------------------------------------------------
# Calculate Ripley's H for all 3 conditions
# -----------------------------------------------------------------------------
max_radius = 500
num_radii = 50

radii_pfa,  pfa_h_curves,  pfa_meta  = collect_ripley_h_from_selected_rois(
    dataset_list=pfa_dataset_list,  selected_regions_for_condition=selected_regions_by_condition["PFA"],
    max_radius=max_radius, num_radii=num_radii)

radii_bp,   bp_h_curves,   bp_meta   = collect_ripley_h_from_selected_rois(
    dataset_list=bp_dataset_list,   selected_regions_for_condition=selected_regions_by_condition["BP"],
    max_radius=max_radius, num_radii=num_radii)

radii_cryo, cryo_h_curves, cryo_meta = collect_ripley_h_from_selected_rois(
    dataset_list=cryo_dataset_list, selected_regions_for_condition=selected_regions_by_condition["Cryo"],
    max_radius=max_radius, num_radii=num_radii)

radii = radii_pfa


# Save Ripley metadata
ripley_roi_metadata = pd.DataFrame(
    [{"condition": "PFA",  **row} for row in pfa_meta]  +
    [{"condition": "BP",   **row} for row in bp_meta]   +
    [{"condition": "Cryo", **row} for row in cryo_meta]
)
ripley_metadata_path = analysis_output_dir / "ripley_roi_metadata.csv"
ripley_roi_metadata.to_csv(ripley_metadata_path, index=False)
print(f"Ripley ROI metadata saved to:\n{ripley_metadata_path}")


# Generate and save plots for all 3 conditions
for condition_name, h_curves, color in [
    ("PFA",  pfa_h_curves,  PFA_COLOR),
    ("BP",   bp_h_curves,   BP_COLOR),
    ("Cryo", cryo_h_curves, CRYO_COLOR)
]:
    plot_ripley_mean_sem_single_condition(radii=radii, h_curve_list=h_curves,
                                          condition_name=condition_name, color=color,
                                          save_dir=analysis_output_dir)
    plot_ripley_median_iqr_single_condition(radii=radii, h_curve_list=h_curves,
                                            condition_name=condition_name, color=color,
                                            save_dir=analysis_output_dir)

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import distance
from sklearn.cluster import DBSCAN
from IPython.display import display


# =============================================================================
# STEP 4 — DBSCAN clustering on selected ROIs (3 conditions: PFA, BP, Cryo)
# =============================================================================

def calculate_dbscan_cluster_metrics(clustered_df):
    labels = clustered_df["labels"]
    unique_labels = set(labels)
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    clustered_points = clustered_df[clustered_df["labels"] != -1]
    fraction_in_cluster = len(clustered_points) / len(clustered_df) if len(clustered_df) > 0 else np.nan
    area_nm2 = (clustered_df["x"].max() - clustered_df["x"].min()) * (clustered_df["y"].max() - clustered_df["y"].min())
    area_um2 = area_nm2 / 1e6
    localization_density = len(clustered_df) / area_um2 if area_um2 > 0 else np.nan
    cluster_diameters = []
    for label in unique_labels:
        if label == -1:
            continue
        cluster_points = clustered_df.loc[clustered_df["labels"] == label, ["x", "y"]].values
        max_distance = distance.pdist(cluster_points).max() if len(cluster_points) > 1 else 0.0
        cluster_diameters.append(max_distance)
    mean_cluster_diameter = np.mean(cluster_diameters) if cluster_diameters else 0.0
    return {
        "n_clusters": n_clusters,
        "fraction_in_cluster": fraction_in_cluster,
        "localization_density": localization_density,
        "mean_cluster_diameter": mean_cluster_diameter
    }


def run_dbscan_on_roi(df_roi, eps, min_samples, max_points=5000):
    clustered_df = df_roi.copy()

    if clustered_df.empty:
        clustered_df["labels"] = pd.Series(dtype=int)
        return clustered_df

    # Wenn zu viele Punkte: zufällig auf max_points reduzieren
    if len(clustered_df) > max_points:
        clustered_df = clustered_df.sample(n=max_points, random_state=42).reset_index(drop=True)

    dbscan = DBSCAN(eps=eps, min_samples=min_samples, algorithm='ball_tree', n_jobs=-1)
    clustered_df.loc[:, "labels"] = dbscan.fit_predict(clustered_df[["x", "y"]])

    return clustered_df


def perform_dbscan_clustering_on_selected_rois(
    dataset_lists_by_condition,
    selected_regions_by_condition,
    eps,
    min_samples,
    save_dir
):
    """
    Perform DBSCAN clustering on all selected ROIs from all conditions.

    Parameters
    ----------
    dataset_lists_by_condition : dict
        {condition_name: [(file_name, dataframe), ...]}
    """
    results_rows = []
    dbscan_results_by_condition = {cond: [] for cond in CONDITION_ORDER}

    dataset_lookup = {
        cond: {file_name: df for file_name, df in ds_list}
        for cond, ds_list in dataset_lists_by_condition.items()
    }

    for condition_name in CONDITION_ORDER:
        dataset_entries = selected_regions_by_condition[condition_name]
        for dataset_idx, dataset_entry in enumerate(dataset_entries, start=1):
            file_name = dataset_entry["file_name"]
            regions = dataset_entry["regions"]
            df = dataset_lookup[condition_name][file_name]
            roi_results = []
            for roi_idx, region in enumerate(regions, start=1):
                df_roi = extract_roi_points(df, region)
                if df_roi.empty:
                    continue
                clustered_df = run_dbscan_on_roi(df_roi=df_roi, eps=eps, min_samples=min_samples)
                metrics = calculate_dbscan_cluster_metrics(clustered_df)
                results_rows.append({
                    "condition": condition_name,
                    "dataset_index": dataset_idx,
                    "file_name": file_name,
                    "roi_index": roi_idx,
                    "eps_nm": eps,
                    "min_samples": min_samples,
                    "n_localizations": len(clustered_df),
                    "n_clusters": metrics["n_clusters"],
                    "fraction_in_cluster": metrics["fraction_in_cluster"],
                    "localization_density": metrics["localization_density"],
                    "mean_cluster_diameter_nm": metrics["mean_cluster_diameter"]
                })
                roi_results.append({"roi_index": roi_idx, "region": region,
                                    "clustered_df": clustered_df, "metrics": metrics})
            dbscan_results_by_condition[condition_name].append({
                "dataset_index": dataset_idx, "file_name": file_name, "roi_results": roi_results
            })

    summary_df = pd.DataFrame(results_rows)
    print("\nDBSCAN clustering results summary:")
    display(summary_df)

    save_name = f"dbscan_summary_eps_{eps:g}_minsamples_{min_samples}.csv"
    save_path = analysis_output_dir / save_name
    summary_df.to_csv(save_path, index=False)
    print(f"\nDBSCAN summary saved to:\n{save_path}")

    return summary_df, dbscan_results_by_condition


# Prompt for DBSCAN parameters
eps = float(input("Enter DBSCAN eps (in nm): ").strip())
min_samples = int(input("Enter DBSCAN min_samples: ").strip())

# Run DBSCAN
dataset_lists_by_condition = {
    "PFA":  pfa_dataset_list,
    "BP":   bp_dataset_list,
    "Cryo": cryo_dataset_list
}

dbscan_summary_df, dbscan_results_by_condition = perform_dbscan_clustering_on_selected_rois(
    dataset_lists_by_condition=dataset_lists_by_condition,
    selected_regions_by_condition=selected_regions_by_condition,
    eps=eps,
    min_samples=min_samples,
    save_dir=analysis_output_dir
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm
import seaborn as sns
from pathlib import Path
from scipy.ndimage import gaussian_filter
import ipywidgets as widgets
from IPython.display import display

# =============================================================================
# CLUSTER OVERLAY ON RECONSTRUCTED SMLM IMAGE
# Renders a 2D density map from all localizations in the ROI,
# then overlays coloured cluster hulls / scatter on top.
# =============================================================================

# Settings
BIN_SIZE_NM      = 10       # pixel size of the density image (nm)
GAUSS_SIGMA_PX   = 1.5      # smoothing sigma for density image (pixels)
CMAP_DENSITY     = "gray_r" # colormap for background density image
CLUSTER_ALPHA    = 0.6      # opacity of cluster points overlay
CLUSTER_SIZE     = 1.2      # point size for cluster overlay
NOISE_COLOR      = "#AAAAAA"
NOISE_SIZE       = 0.3
NOISE_ALPHA      = 0.25
SHOW_HULLS       = True     # draw convex hull outline around each cluster
HULL_LW          = 0.6
SHOW_LABELS      = True
LABEL_FONTSIZE   = 4
CMAP_CLUSTERS    = "tab20"
DPI              = 300

vis_output_dir = analysis_output_dir / "cluster_overlays"
vis_output_dir.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Build ROI lookup (same as before)
# -----------------------------------------------------------------------------

roi_lookup = {}
for condition_name in CONDITION_ORDER:
    for ds_entry in dbscan_results_by_condition[condition_name]:
        ds_idx    = ds_entry["dataset_index"]
        file_name = ds_entry["file_name"]
        for roi_res in ds_entry["roi_results"]:
            roi_idx = roi_res["roi_index"]
            n_clust = len([l for l in roi_res["clustered_df"]["labels"].unique() if l != -1])
            key = (f"{condition_name}  |  Dataset {ds_idx}  |  ROI {roi_idx}"
                   f"  |  {n_clust} clusters  |  {file_name}")
            roi_lookup[key] = {
                "roi_res":        roi_res,
                "condition_name": condition_name,
                "ds_idx":         ds_idx,
                "roi_idx":        roi_idx,
            }

all_keys = sorted(roi_lookup.keys())


# -----------------------------------------------------------------------------
# Core plot function
# -----------------------------------------------------------------------------

def make_density_image(x, y, bin_size_nm, sigma_px):
    """Bin localizations into a 2D histogram and smooth."""
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()
    nx = max(int((xmax - xmin) / bin_size_nm) + 1, 1)
    ny = max(int((ymax - ymin) / bin_size_nm) + 1, 1)
    hist, xedges, yedges = np.histogram2d(
        x, y,
        bins=[nx, ny],
        range=[[xmin, xmax], [ymin, ymax]]
    )
    hist = gaussian_filter(hist.astype(float), sigma=sigma_px)
    return hist, xedges, yedges


def convex_hull_path(pts):
    """Return hull vertices for plotting, or None if < 3 points."""
    from scipy.spatial import ConvexHull
    if len(pts) < 3:
        return None
    try:
        hull = ConvexHull(pts)
        verts = pts[hull.vertices]
        verts = np.vstack([verts, verts[0]])  # close the loop
        return verts
    except Exception:
        return None


def plot_overlay(entry, save=True):
    roi_res        = entry["roi_res"]
    condition_name = entry["condition_name"]
    ds_idx         = entry["ds_idx"]
    roi_idx        = entry["roi_idx"]

    clustered_df  = roi_res["clustered_df"]
    labels        = clustered_df["labels"].values
    x             = clustered_df["x"].values
    y             = clustered_df["y"].values

    unique_labels = sorted([l for l in np.unique(labels) if l != -1])
    n_clusters    = len(unique_labels)
    cmap_obj      = cm.get_cmap(CMAP_CLUSTERS, max(n_clusters, 1))
    colors        = {label: cmap_obj(i % cmap_obj.N)
                     for i, label in enumerate(unique_labels)}

    # ── build density background ─────────────────────────────────────────
    hist, xedges, yedges = make_density_image(
        x, y, BIN_SIZE_NM, GAUSS_SIGMA_PX)

    fig, ax = plt.subplots(figsize=(6, 6))

    # density image — imshow expects (row=y, col=x), transposed
    ax.imshow(
        hist.T,
        origin="lower",
        extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
        cmap=CMAP_DENSITY,
        norm=LogNorm(vmin=max(hist[hist > 0].min(), 0.1), vmax=hist.max()),
        aspect="equal",
        interpolation="bilinear",
    )

    # ── noise points ─────────────────────────────────────────────────────
    noise_mask = labels == -1
    if noise_mask.any():
        ax.scatter(x[noise_mask], y[noise_mask],
                   c=NOISE_COLOR, s=NOISE_SIZE,
                   alpha=NOISE_ALPHA, rasterized=True, zorder=2)

    # ── cluster points + hulls + labels ──────────────────────────────────
    for label in unique_labels:
        mask  = labels == label
        pts   = np.column_stack([x[mask], y[mask]])
        color = colors[label]

        # scatter
        ax.scatter(x[mask], y[mask],
                   c=[color], s=CLUSTER_SIZE,
                   alpha=CLUSTER_ALPHA, rasterized=True, zorder=3)

        # convex hull outline
        if SHOW_HULLS:
            hull_verts = convex_hull_path(pts)
            if hull_verts is not None:
                ax.plot(hull_verts[:, 0], hull_verts[:, 1],
                        color=color, linewidth=HULL_LW,
                        alpha=0.85, zorder=4)

        # centroid label
        if SHOW_LABELS:
            cx, cy = pts[:, 0].mean(), pts[:, 1].mean()
            ax.text(cx, cy, str(label),
                    fontsize=LABEL_FONTSIZE, ha="center", va="center",
                    color="white", fontweight="bold", zorder=5,
                    bbox=dict(boxstyle="round,pad=0.1",
                              fc=color, ec="none", alpha=0.75))

    ax.set_xlim(xedges[0], xedges[-1])
    ax.set_ylim(yedges[0], yedges[-1])
    ax.set_title(
        f"{condition_name}  |  Dataset {ds_idx}  |  ROI {roi_idx}\n"
        f"{n_clusters} clusters  |  eps={eps} nm  |  minPts={min_samples}",
        fontsize=8
    )
    ax.set_xlabel("x (nm)", fontsize=7)
    ax.set_ylabel("y (nm)", fontsize=7)
    ax.tick_params(labelsize=6)
    sns.despine(ax=ax)
    plt.tight_layout()

    if save:
        fname = f"{condition_name}_ds{ds_idx}_roi{roi_idx}_overlay.png"
        fig.savefig(vis_output_dir / fname, dpi=DPI, bbox_inches="tight")
        print(f"Saved: {vis_output_dir / fname}")

    plt.show()
    plt.close(fig)


# -----------------------------------------------------------------------------
# Widget
# -----------------------------------------------------------------------------

select_widget = widgets.SelectMultiple(
    options = all_keys,
    layout  = widgets.Layout(width="900px", height="280px"),
    style   = {"description_width": "0px"},
)

btn_plot = widgets.Button(
    description="Plot selected", button_style="primary",
    layout=widgets.Layout(width="180px"))
btn_all = widgets.Button(
    description="Plot ALL", button_style="warning",
    layout=widgets.Layout(width="120px"))
btn_save = widgets.ToggleButton(
    value=True, description="Save to disk", button_style="success",
    layout=widgets.Layout(width="140px"))

out = widgets.Output()

display(widgets.HTML("<b>Select ROIs to overlay (Ctrl+click for multiple):</b>"))
display(select_widget)
display(widgets.HBox([btn_plot, btn_all, btn_save]))
display(out)


def on_plot_selected(b):
    with out:
        out.clear_output()
        selected = select_widget.value
        if not selected:
            print("No ROIs selected.")
            return
        for key in selected:
            plot_overlay(roi_lookup[key], save=btn_save.value)

def on_plot_all(b):
    with out:
        out.clear_output()
        for key in all_keys:
            plot_overlay(roi_lookup[key], save=btn_save.value)

btn_plot.on_click(on_plot_selected)
btn_all.on_click(on_plot_all)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

# =============================================================================
# CLUSTER VISUALISATION — interactive ROI picker
# =============================================================================

# Settings
NOISE_COLOR    = "#CCCCCC"
NOISE_SIZE     = 0.5
CLUSTER_SIZE   = 1.5
SHOW_LABELS    = True
LABEL_FONTSIZE = 4
DPI            = 300
CMAP           = "tab20"

vis_output_dir = analysis_output_dir / "cluster_visualisations"
vis_output_dir.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Build a lookup of all available ROIs
# -----------------------------------------------------------------------------

roi_lookup = {}   # key: "PFA | ds1 | ROI 1 | filename.csv" -> roi_res dict

for condition_name in CONDITION_ORDER:
    for ds_entry in dbscan_results_by_condition[condition_name]:
        ds_idx    = ds_entry["dataset_index"]
        file_name = ds_entry["file_name"]
        for roi_res in ds_entry["roi_results"]:
            roi_idx = roi_res["roi_index"]
            n_clust = len(roi_res["clustered_df"]["labels"].unique()) - 1
            key = (f"{condition_name}  |  Dataset {ds_idx}  |  ROI {roi_idx}"
                   f"  |  {n_clust} clusters  |  {file_name}")
            roi_lookup[key] = {
                "roi_res":        roi_res,
                "condition_name": condition_name,
                "ds_idx":         ds_idx,
                "roi_idx":        roi_idx,
            }

all_keys = sorted(roi_lookup.keys())
print(f"Found {len(all_keys)} ROIs across all conditions.\n")


# -----------------------------------------------------------------------------
# Widget: multi-select ROI list + buttons
# -----------------------------------------------------------------------------

select_widget = widgets.SelectMultiple(
    options    = all_keys,
    description= "",
    layout     = widgets.Layout(width="900px", height="300px"),
    style      = {"description_width": "0px"},
)

btn_plot = widgets.Button(
    description  = "Plot selected ROIs",
    button_style = "primary",
    layout       = widgets.Layout(width="200px")
)
btn_all = widgets.Button(
    description  = "Plot ALL ROIs",
    button_style = "warning",
    layout       = widgets.Layout(width="200px")
)
btn_save = widgets.ToggleButton(
    value        = True,
    description  = "Save to disk",
    button_style = "success",
    layout       = widgets.Layout(width="150px")
)

out = widgets.Output()

display(widgets.HTML("<b>Select one or more ROIs (Ctrl+click for multiple):</b>"))
display(select_widget)
display(widgets.HBox([btn_plot, btn_all, btn_save]))
display(out)


# -----------------------------------------------------------------------------
# Plot function
# -----------------------------------------------------------------------------

def plot_roi_clusters(entry, save=True):
    roi_res        = entry["roi_res"]
    condition_name = entry["condition_name"]
    ds_idx         = entry["ds_idx"]
    roi_idx        = entry["roi_idx"]

    clustered_df  = roi_res["clustered_df"]
    labels        = clustered_df["labels"].values
    x             = clustered_df["x"].values
    y             = clustered_df["y"].values

    unique_labels = sorted([l for l in np.unique(labels) if l != -1])
    n_clusters    = len(unique_labels)
    cmap_obj      = cm.get_cmap(CMAP, max(n_clusters, 1))
    colors        = {label: cmap_obj(i % cmap_obj.N)
                     for i, label in enumerate(unique_labels)}

    fig, ax = plt.subplots(figsize=(5, 5))

    # Noise
    noise_mask = labels == -1
    if noise_mask.any():
        ax.scatter(x[noise_mask], y[noise_mask],
                   c=NOISE_COLOR, s=NOISE_SIZE, alpha=0.4, rasterized=True)

    # Clusters
    for label in unique_labels:
        mask = labels == label
        ax.scatter(x[mask], y[mask],
                   c=[colors[label]], s=CLUSTER_SIZE,
                   alpha=0.85, rasterized=True)
        if SHOW_LABELS:
            cx, cy = x[mask].mean(), y[mask].mean()
            ax.text(cx, cy, str(label),
                    fontsize=LABEL_FONTSIZE, ha="center", va="center",
                    color="black", fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.1", fc="white",
                              ec="none", alpha=0.6))

    ax.set_aspect("equal")
    ax.set_title(
        f"{condition_name}  |  Dataset {ds_idx}  |  ROI {roi_idx}\n"
        f"{n_clusters} clusters  |  eps={eps} nm  |  minPts={min_samples}",
        fontsize=7
    )
    ax.set_xlabel("x (nm)", fontsize=6)
    ax.set_ylabel("y (nm)", fontsize=6)
    ax.tick_params(labelsize=5)
    sns.despine(ax=ax)
    plt.tight_layout()

    if save:
        fname = f"{condition_name}_ds{ds_idx}_roi{roi_idx}_clusters.png"
        fig.savefig(vis_output_dir / fname, dpi=DPI, bbox_inches="tight")
        print(f"Saved: {vis_output_dir / fname}")

    plt.show()
    plt.close(fig)


# -----------------------------------------------------------------------------
# Button callbacks
# -----------------------------------------------------------------------------

def on_plot_selected(b):
    with out:
        out.clear_output()
        selected = select_widget.value
        if not selected:
            print("No ROIs selected — Ctrl+click to select one or more.")
            return
        for key in selected:
            plot_roi_clusters(roi_lookup[key], save=btn_save.value)

def on_plot_all(b):
    with out:
        out.clear_output()
        for key in all_keys:
            plot_roi_clusters(roi_lookup[key], save=btn_save.value)

btn_plot.on_click(on_plot_selected)
btn_all.on_click(on_plot_all)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
import seaborn as sns
from itertools import combinations

# =============================================================================
# STEP 5 — Plot DBSCAN summary metrics for PFA vs BP vs Cryo
# =============================================================================

sns.set_style("white")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 14
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def compute_metric_summary_stats(df, metric_col, condition_name):
    vals = pd.to_numeric(df.loc[df["condition"] == condition_name, metric_col], errors="coerce").dropna()
    if len(vals) == 0:
        return {"condition": condition_name, "metric": metric_col, "n": 0,
                "mean": np.nan, "median": np.nan, "std": np.nan, "sem": np.nan,
                "min": np.nan, "q25": np.nan, "q75": np.nan, "iqr": np.nan, "max": np.nan}
    q25 = np.percentile(vals, 25)
    q75 = np.percentile(vals, 75)
    return {
        "condition": condition_name, "metric": metric_col, "n": int(len(vals)),
        "mean": float(np.mean(vals)), "median": float(np.median(vals)),
        "std": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        "sem": float(np.std(vals, ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else 0.0,
        "min": float(np.min(vals)), "q25": float(q25), "q75": float(q75),
        "iqr": float(q75 - q25), "max": float(np.max(vals))
    }


# Plot specifications
plot_specs = [
    {"metric": "n_clusters", "title": "Number of Clusters per ROI",
     "ylabel": "Number of clusters", "plot_type": "violin",
     "filename": f"dbscan_n_clusters_PFA_BP_Cryo_eps_{eps:g}_minsamples_{min_samples}.png"},
    {"metric": "fraction_in_cluster", "title": "Fraction of Localizations in Clusters",
     "ylabel": "Fraction in cluster", "plot_type": "box",
     "filename": f"dbscan_fraction_in_cluster_PFA_BP_Cryo_eps_{eps:g}_minsamples_{min_samples}.png"},
    {"metric": "mean_cluster_diameter_nm", "title": "Mean Cluster Diameter",
     "ylabel": "Mean cluster diameter (nm)", "plot_type": "violin",
     "filename": f"dbscan_mean_cluster_diameter_PFA_BP_Cryo_eps_{eps:g}_minsamples_{min_samples}.png"},
    {"metric": "localization_density", "title": "Localization Density",
     "ylabel": "Localization density (locs / µm²)", "plot_type": "box",
     "filename": f"dbscan_localization_density_PFA_BP_Cryo_eps_{eps:g}_minsamples_{min_samples}.png"}
]


# Build summary statistics and pairwise comparison tables
summary_stats_rows = []
comparison_rows = []

for spec in plot_specs:
    metric = spec["metric"]
    for condition_name in CONDITION_ORDER:
        summary_stats_rows.append(compute_metric_summary_stats(dbscan_summary_df, metric, condition_name))

    # All pairwise comparisons between 3 conditions
    for cond_a, cond_b in combinations(CONDITION_ORDER, 2):
        stat, p_value = compare_two_conditions_mannwhitney(dbscan_summary_df, metric, cond_a, cond_b)
        vals_a = pd.to_numeric(dbscan_summary_df.loc[dbscan_summary_df["condition"] == cond_a, metric], errors="coerce").dropna()
        vals_b = pd.to_numeric(dbscan_summary_df.loc[dbscan_summary_df["condition"] == cond_b, metric], errors="coerce").dropna()
        comparison_rows.append({
            "metric": metric,
            "condition_a": cond_a, "condition_b": cond_b,
            "n_a": int(len(vals_a)), "n_b": int(len(vals_b)),
            "mannwhitney_u_statistic": stat,
            "p_value": p_value,
            "significance": p_to_stars(p_value)
        })

dbscan_summary_stats_df = pd.DataFrame(summary_stats_rows)
dbscan_comparison_stats_df = pd.DataFrame(comparison_rows)

summary_stats_path = analysis_output_dir / f"dbscan_summary_stats_by_condition_eps_{eps:g}_minsamples_{min_samples}.csv"
comparison_stats_path = analysis_output_dir / f"dbscan_condition_comparisons_eps_{eps:g}_minsamples_{min_samples}.csv"
dbscan_summary_stats_df.to_csv(summary_stats_path, index=False)
dbscan_comparison_stats_df.to_csv(comparison_stats_path, index=False)
print(f"Summary statistics saved to:\n{summary_stats_path}")
print(f"Condition comparison statistics saved to:\n{comparison_stats_path}")


# Create each plot
for spec in plot_specs:
    metric = spec["metric"]
    title = spec["title"]
    ylabel = spec["ylabel"]
    plot_type = spec["plot_type"]
    filename = spec["filename"]

    plot_df = dbscan_summary_df.copy()
    plot_df = plot_df[plot_df["condition"].isin(CONDITION_ORDER)].copy()
    plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")
    plot_df = plot_df.dropna(subset=[metric])

    fig, ax = plt.subplots(figsize=(6.5, 6))

    if plot_type == "violin":
        sns.violinplot(data=plot_df, x="condition", y=metric, hue="condition",
                       order=CONDITION_ORDER, hue_order=CONDITION_ORDER, palette=PALETTE,
                       inner=None, cut=0, linewidth=1.2, dodge=False, legend=False, ax=ax)
        add_violin_median_lines(ax, plot_df, metric)
        sns.swarmplot(data=plot_df, x="condition", y=metric, order=CONDITION_ORDER,
                      color="black", size=3, alpha=0.8, ax=ax)

    elif plot_type == "box":
        sns.boxplot(data=plot_df, x="condition", y=metric, hue="condition",
                    order=CONDITION_ORDER, hue_order=CONDITION_ORDER, palette=PALETTE,
                    width=0.5, fliersize=0, linewidth=1.2, dodge=False, legend=False, ax=ax)
        sns.swarmplot(data=plot_df, x="condition", y=metric, order=CONDITION_ORDER,
                      color="black", size=3, alpha=0.8, ax=ax)

    ax.set_title(title, fontsize=16)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=14)
    ax.tick_params(axis="x", labelsize=12)
    ax.tick_params(axis="y", labelsize=12)
    style_condition_ticklabels(ax)

    if metric == "fraction_in_cluster":
        vals = plot_df[metric].dropna()
        if len(vals) > 0:
            ax.set_ylim(np.floor(vals.min() * 20) / 20, 1.0)

    add_significance_annotation_3groups(ax, plot_df, metric)

    plt.show()
    save_path = analysis_output_dir / filename
    fig.savefig(save_path, dpi=600, bbox_inches="tight")
    print(f"Figure saved to:\n{save_path}")
    plt.close(fig)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import KDTree, distance
from scipy.stats import sem as scipy_sem

# =============================================================================
# STEP 5E — Radial Distribution Function (pair correlation g(r))
#            + Nearest-neighbour distances between cluster centroids
# =============================================================================

# ---------------------------------------------------------------------------
# Helper: compute cluster centroids from a clustered_df
# ---------------------------------------------------------------------------

def get_cluster_centroids(clustered_df):
    centroids = []
    for label in clustered_df["labels"].unique():
        if label == -1:
            continue
        pts = clustered_df.loc[clustered_df["labels"] == label, ["x", "y"]].values
        centroids.append(pts.mean(axis=0))
    return np.array(centroids) if centroids else np.empty((0, 2))


# ---------------------------------------------------------------------------
# 1. Pair correlation function g(r)
# ---------------------------------------------------------------------------

def compute_pcf(df_roi, max_radius, n_bins=50, max_points=3000):
    pts = df_roi[["x", "y"]].values
    n   = len(pts)
    if n < 5:
        return None, None
    if n > max_points:
        idx = np.random.choice(n, max_points, replace=False)
        pts = pts[idx]
        n   = max_points
    area = (df_roi["x"].max() - df_roi["x"].min()) * \
           (df_roi["y"].max() - df_roi["y"].min())
    if area == 0:
        return None, None
    pw = distance.pdist(pts)
    pw = pw[pw <= max_radius]
    bin_edges  = np.linspace(0, max_radius, n_bins + 1)
    bin_widths = np.diff(bin_edges)
    bin_mids   = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    counts, _  = np.histogram(pw, bins=bin_edges)
    counts     = counts * 2
    lambda_hat  = n / area
    csr_expected = lambda_hat * 2 * np.pi * bin_mids * bin_widths * n
    with np.errstate(invalid="ignore", divide="ignore"):
        g = np.where(csr_expected > 0, counts / csr_expected, np.nan)
    return bin_mids, g


def collect_pcf_curves(dataset_list, selected_regions_for_condition,
                       max_radius, n_bins=50):
    curves = []
    dataset_lookup = {fn: df for fn, df in dataset_list}
    for ds_entry in selected_regions_for_condition:
        fn  = ds_entry["file_name"]
        df  = dataset_lookup.get(fn)
        if df is None:
            continue
        for region in ds_entry["regions"]:
            df_roi = extract_roi_points(df, region)
            if len(df_roi) < 5:
                continue
            r, g = compute_pcf(df_roi, max_radius=max_radius, n_bins=n_bins)
            if r is not None:
                curves.append(g)
    return curves


# --- compute for all conditions ---
pcf_max_radius = max_radius
pcf_n_bins     = 60

pcf_curves = {}
for condition_name, ds_list, sel_regions in [
    ("PFA",  pfa_dataset_list,  selected_regions_by_condition["PFA"]),
    ("BP",   bp_dataset_list,   selected_regions_by_condition["BP"]),
    ("Cryo", cryo_dataset_list, selected_regions_by_condition["Cryo"]),
]:
    pcf_curves[condition_name] = collect_pcf_curves(
        ds_list, sel_regions,
        max_radius=pcf_max_radius, n_bins=pcf_n_bins
    )
    print(f"{condition_name}: {len(pcf_curves[condition_name])} g(r) curves computed")

bin_mids_ref = np.linspace(0, pcf_max_radius, pcf_n_bins + 1)
bin_mids_ref = 0.5 * (bin_mids_ref[:-1] + bin_mids_ref[1:])


# --- plot: individual conditions ---
for condition_name in CONDITION_ORDER:
    curves = pcf_curves[condition_name]
    if not curves:
        continue
    color  = PALETTE[condition_name]
    arr    = np.vstack(curves)
    mean_g = np.nanmean(arr, axis=0)
    sem_g  = scipy_sem(arr, axis=0, nan_policy="omit")

    fig, ax = plt.subplots(figsize=(7, 5))
    for c in curves:
        ax.plot(bin_mids_ref, c, color="0.75", linewidth=0.8, alpha=0.7)
    ax.plot(bin_mids_ref, mean_g, color=color, linewidth=2.5,
            label=f"Mean g(r) — {condition_name}")
    ax.fill_between(bin_mids_ref, mean_g - sem_g, mean_g + sem_g,
                    color=color, alpha=0.20, label="SEM")
    ax.axhline(1, color="black", linestyle="--", linewidth=1.2,
               label="g(r) = 1  (CSR)")
    ax.set_xlim(0, pcf_max_radius)
    ax.set_xlabel("Radius r (nm)", fontsize=13)
    ax.set_ylabel("Pair correlation g(r)", fontsize=13)
    ax.set_title(f"Pair Correlation Function — {condition_name}", fontsize=14)
    ax.legend(frameon=False, fontsize=9)
    sns.despine(ax=ax)
    save_path = analysis_output_dir / f"pcf_g_r_{condition_name}.png"
    fig.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"Saved:\n{save_path}")


# --- plot: all conditions overlaid ---
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.axhline(1, color="black", linestyle="--", linewidth=1.2, label="CSR")
for condition_name in CONDITION_ORDER:
    curves = pcf_curves[condition_name]
    if not curves:
        continue
    color  = PALETTE[condition_name]
    arr    = np.vstack(curves)
    mean_g = np.nanmean(arr, axis=0)
    sem_g  = scipy_sem(arr, axis=0, nan_policy="omit")
    ax.plot(bin_mids_ref, mean_g, color=color, linewidth=2.2,
            label=condition_name)
    ax.fill_between(bin_mids_ref, mean_g - sem_g, mean_g + sem_g,
                    color=color, alpha=0.18)
ax.set_xlim(0, pcf_max_radius)
ax.set_xlabel("Radius r (nm)", fontsize=13)
ax.set_ylabel("Pair correlation g(r)", fontsize=13)
ax.set_title("Pair Correlation Function — All Conditions", fontsize=14)
ax.legend(frameon=False, fontsize=10)
sns.despine(ax=ax)
save_path = analysis_output_dir / "pcf_g_r_all_conditions_overlay.png"
fig.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved:\n{save_path}")


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import KDTree

# =============================================================================
# STEP 5F — K-nearest-neighbour distances between cluster centroids (k=3)
# =============================================================================
# For every cluster centroid we find the 3 nearest other centroids and
# take the mean of those 3 distances. This is more robust than single NN
# because one isolated cluster won't dominate the result.

K_NEIGHBOURS = 3

def collect_knn_distances(dbscan_results_by_condition, k=3):
    rows = []
    for condition_name in CONDITION_ORDER:
        for ds_entry in dbscan_results_by_condition[condition_name]:
            ds_idx    = ds_entry["dataset_index"]
            file_name = ds_entry["file_name"]
            for roi_res in ds_entry["roi_results"]:
                roi_idx      = roi_res["roi_index"]
                clustered_df = roi_res["clustered_df"]
                centroids    = get_cluster_centroids(clustered_df)

                # need at least k+1 clusters (k neighbours + self)
                if len(centroids) < k + 1:
                    continue

                tree = KDTree(centroids)
                # k+1 because first result is always self (distance=0)
                knn_dists, _ = tree.query(centroids, k=k + 1)
                knn_dists    = knn_dists[:, 1:]   # drop self column

                for i, dists in enumerate(knn_dists):
                    rows.append({
                        "condition":          condition_name,
                        "dataset_index":      ds_idx,
                        "file_name":          file_name,
                        "roi_index":          roi_idx,
                        "centroid_index":     i,
                        "mean_knn_dist_nm":   float(dists.mean()),
                        "nn1_dist_nm":        float(dists[0]),
                        "nn2_dist_nm":        float(dists[1]),
                        "nn3_dist_nm":        float(dists[2]),
                    })
    return pd.DataFrame(rows)


knn_df = collect_knn_distances(dbscan_results_by_condition, k=K_NEIGHBOURS)

knn_path = analysis_output_dir / f"centroid_knn3_distances_eps_{eps:g}_minsamples_{min_samples}.csv"
knn_df.to_csv(knn_path, index=False)
print(f"K=3 NN distances saved to:\n{knn_path}")

# per-ROI median for comparison plot
knn_per_roi = (
    knn_df.groupby(["condition", "dataset_index", "file_name", "roi_index"])
    ["mean_knn_dist_nm"].median().reset_index()
    .rename(columns={"mean_knn_dist_nm": "median_mean_knn_dist_nm"})
)

# --- violin plot ---
fig, ax = plt.subplots(figsize=(6.5, 6))
plot_knn = knn_per_roi[knn_per_roi["condition"].isin(CONDITION_ORDER)].copy()
plot_knn["median_mean_knn_dist_nm"] = pd.to_numeric(
    plot_knn["median_mean_knn_dist_nm"], errors="coerce")
plot_knn = plot_knn.dropna(subset=["median_mean_knn_dist_nm"])

sns.violinplot(
    data=plot_knn, x="condition", y="median_mean_knn_dist_nm", hue="condition",
    order=CONDITION_ORDER, hue_order=CONDITION_ORDER, palette=PALETTE,
    inner=None, cut=0, linewidth=1.2, dodge=False, legend=False, ax=ax
)
add_violin_median_lines(ax, plot_knn, "median_mean_knn_dist_nm")
sns.swarmplot(
    data=plot_knn, x="condition", y="median_mean_knn_dist_nm",
    order=CONDITION_ORDER, color="black", size=3, alpha=0.8, ax=ax
)
ax.set_title(f"Mean Distance to {K_NEIGHBOURS} Nearest Cluster Centroids", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel(f"Median mean k={K_NEIGHBOURS} NN distance (nm)", fontsize=12)
style_condition_ticklabels(ax)
add_significance_annotation_3groups(ax, plot_knn, "median_mean_knn_dist_nm")
sns.despine(ax=ax)
save_path = analysis_output_dir / f"centroid_knn3_violin_eps_{eps:g}_minsamples_{min_samples}.png"
fig.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved:\n{save_path}")

# --- histogram: pooled distribution ---
fig, ax = plt.subplots(figsize=(7, 5))
for condition_name in CONDITION_ORDER:
    vals = knn_df.loc[knn_df["condition"] == condition_name, "mean_knn_dist_nm"].dropna()
    if len(vals) == 0:
        continue
    ax.hist(vals, bins=40, alpha=0.50, color=PALETTE[condition_name],
            label=condition_name, density=True)
    ax.axvline(vals.median(), color=PALETTE[condition_name],
               linewidth=2.0, linestyle="--")
ax.set_xlabel(f"Mean k={K_NEIGHBOURS} NN distance (nm)", fontsize=13)
ax.set_ylabel("Density", fontsize=13)
ax.set_title(f"Distribution of Mean k={K_NEIGHBOURS} NN Distances Between Cluster Centroids", fontsize=13)
ax.legend(frameon=False, fontsize=10)
sns.despine(ax=ax)
save_path = analysis_output_dir / f"centroid_knn3_histogram_eps_{eps:g}_minsamples_{min_samples}.png"
fig.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved:\n{save_path}")

print(f"\nStep 5F complete — k={K_NEIGHBOURS} NN centroid analysis done.")

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import ConvexHull

# =============================================================================
# STEP 5B — Per-cluster metrics (size, area, radius, intra-cluster density)
# =============================================================================

def convex_hull_area(points):
    if len(points) < 3:
        return 0.0
    try:
        return ConvexHull(points).volume
    except Exception:
        return 0.0


def extract_per_cluster_metrics(clustered_df):
    rows = []
    for label in clustered_df["labels"].unique():
        if label == -1:
            continue
        pts = clustered_df.loc[clustered_df["labels"] == label, ["x", "y"]].values
        n_locs    = len(pts)
        area_nm2  = convex_hull_area(pts)
        area_um2  = area_nm2 / 1e6
        radius_nm = np.sqrt(area_nm2 / np.pi) if area_nm2 > 0 else 0.0
        intra_density = n_locs / area_um2 if area_um2 > 0 else np.nan
        rows.append({
            "label":                       int(label),
            "size_locs":                   n_locs,
            "area_nm2":                    area_nm2,
            "area_um2":                    area_um2,
            "radius_nm":                   radius_nm,
            "intra_density_locs_per_um2":  intra_density,
        })
    return pd.DataFrame(rows)


def collect_cluster_size_density_metrics(dbscan_results_by_condition):
    per_cluster_rows = []
    per_roi_rows     = []

    for condition_name in CONDITION_ORDER:
        for ds_entry in dbscan_results_by_condition[condition_name]:
            ds_idx    = ds_entry["dataset_index"]
            file_name = ds_entry["file_name"]
            for roi_res in ds_entry["roi_results"]:
                roi_idx      = roi_res["roi_index"]
                clustered_df = roi_res["clustered_df"]
                metrics_df   = extract_per_cluster_metrics(clustered_df)

                if not metrics_df.empty:
                    for _, row in metrics_df.iterrows():
                        per_cluster_rows.append({
                            "condition":      condition_name,
                            "dataset_index":  ds_idx,
                            "file_name":      file_name,
                            "roi_index":      roi_idx,
                            **row.to_dict()
                        })
                    per_roi_rows.append({
                        "condition":        condition_name,
                        "dataset_index":    ds_idx,
                        "file_name":        file_name,
                        "roi_index":        roi_idx,
                        "n_clusters":       len(metrics_df),
                        "mean_size_locs":   metrics_df["size_locs"].mean(),
                        "median_size_locs": metrics_df["size_locs"].median(),
                        "mean_area_um2":    metrics_df["area_um2"].mean(),
                        "mean_radius_nm":   metrics_df["radius_nm"].mean(),
                        "median_radius_nm": metrics_df["radius_nm"].median(),
                        "mean_intra_density": metrics_df["intra_density_locs_per_um2"].mean(),
                    })

    return pd.DataFrame(per_cluster_rows), pd.DataFrame(per_roi_rows)


per_cluster_df, per_roi_cluster_df = collect_cluster_size_density_metrics(dbscan_results_by_condition)

print(f"Total clusters found: {len(per_cluster_df)}")
print(f"Per-ROI summaries: {len(per_roi_cluster_df)}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# Localizations per cluster — violin + box with whiskers (two separate plots)
# =============================================================================

plot_lpc = per_cluster_df[per_cluster_df["condition"].isin(CONDITION_ORDER)].copy()
plot_lpc["size_locs"] = pd.to_numeric(plot_lpc["size_locs"], errors="coerce")
plot_lpc = plot_lpc.dropna(subset=["size_locs"])

lpc_per_roi = (
    plot_lpc.groupby(["condition", "dataset_index", "file_name", "roi_index"])
    ["size_locs"].median().reset_index()
    .rename(columns={"size_locs": "median_locs_per_cluster"})
)

# ---------------------------------------------------------------------------
# Plot 1 — Violin
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.5, 6))
sns.violinplot(
    data=lpc_per_roi, x="condition", y="median_locs_per_cluster", hue="condition",
    order=CONDITION_ORDER, hue_order=CONDITION_ORDER, palette=PALETTE,
    inner=None, cut=0, linewidth=1.2, dodge=False, legend=False, ax=ax
)
add_violin_median_lines(ax, lpc_per_roi, "median_locs_per_cluster")
sns.swarmplot(
    data=lpc_per_roi, x="condition", y="median_locs_per_cluster",
    order=CONDITION_ORDER, color="black", size=3, alpha=0.8, ax=ax
)
ax.set_title("Localizations per Cluster", fontsize=16)
ax.set_xlabel("")
ax.set_ylabel("Median localizations per cluster", fontsize=14)
style_condition_ticklabels(ax)
add_significance_annotation_3groups(ax, lpc_per_roi, "median_locs_per_cluster")
sns.despine(ax=ax)
save_path = analysis_output_dir / f"locs_per_cluster_violin_eps_{eps:g}_minsamples_{min_samples}.png"
fig.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved:\n{save_path}")

# ---------------------------------------------------------------------------
# Plot 2 — Box with whiskers extending to min/max
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.5, 6))

# draw box plot with whiskers to min/max (whis=[0, 100])
sns.boxplot(
    data=lpc_per_roi, x="condition", y="median_locs_per_cluster", hue="condition",
    order=CONDITION_ORDER, hue_order=CONDITION_ORDER, palette=PALETTE,
    width=0.5, linewidth=1.2, dodge=False, legend=False,
    whis=[0, 100],      # whiskers extend to min and max
    fliersize=0,        # hide default outlier dots since whiskers cover full range
    ax=ax
)
sns.swarmplot(
    data=lpc_per_roi, x="condition", y="median_locs_per_cluster",
    order=CONDITION_ORDER, color="black", size=4, alpha=0.85, ax=ax
)
ax.set_title("Localizations per Cluster", fontsize=16)
ax.set_xlabel("")
ax.set_ylabel("Median localizations per cluster", fontsize=14)
style_condition_ticklabels(ax)
add_significance_annotation_3groups(ax, lpc_per_roi, "median_locs_per_cluster")
sns.despine(ax=ax)
save_path = analysis_output_dir / f"locs_per_cluster_box_eps_{eps:g}_minsamples_{min_samples}.png"
fig.savefig(save_path, dpi=600, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved:\n{save_path}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import sem as scipy_sem
from scipy.stats import mannwhitneyu
from itertools import combinations

# =============================================================================
# FIGURE RECREATION CELL
# Adjust all styling here — then re-run this single cell to update all plots.
# All data comes from variables already computed in the cells above.
# =============================================================================

# -----------------------------------------------------------------------------
# ❶  STYLE SETTINGS — edit these
# -----------------------------------------------------------------------------

# Colors
FIG_PALETTE = {
    "PFA":  "#481E52",
    "BP":   "#A7C946",
    "Cryo": "#4A979F",
}
INDIVIDUAL_LINE_COLOR = "0.80"   # grey for individual Ripley curves

# Font sizes
TITLE_FONTSIZE  = 20
LABEL_FONTSIZE  = 17
TICK_FONTSIZE   = 17
ANNOT_FONTSIZE  = 15
LEGEND_FONTSIZE = 12

# Line widths
MEAN_LW    = 2.2    # main mean/median line
SEM_ALPHA  = 0.20   # shading opacity
INDIV_LW   = 0.8    # individual Ripley curves
REF_LW     = 1.2    # dashed reference line (H=0, g=1)
BOX_LW     = 1.2    # box/violin border
BRACKET_LW = 0.8    # significance bracket lines

# Swarm / box sizes
SWARM_SIZE = 5
BOX_WIDTH  = 0.5
VIOLIN_CUT = 0

# Figure size (inches)
FIG_W = 6.5
FIG_H = 6

# DPI for saved files
SAVE_DPI = 600

# Save directory
save_dir = analysis_output_dir


# -----------------------------------------------------------------------------
# ❷  SHARED HELPERS
# -----------------------------------------------------------------------------

def p_to_stars(p):
    if p < 0.0001: return "****"
    elif p < 0.001: return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    else:           return "ns"


def add_sig_brackets(ax, df, metric, condition_order=CONDITION_ORDER):
    pairs    = list(combinations(condition_order, 2))
    vals_all = pd.to_numeric(df[metric], errors="coerce").dropna()
    if vals_all.empty:
        return
    y_max   = vals_all.max()
    y_range = vals_all.max() - vals_all.min()
    offsets = [0.10, 0.20, 0.30]
    x_pos   = {c: i for i, c in enumerate(condition_order)}
    for (ca, cb), off in zip(pairs, offsets):
        va = pd.to_numeric(df.loc[df["condition"] == ca, metric], errors="coerce").dropna()
        vb = pd.to_numeric(df.loc[df["condition"] == cb, metric], errors="coerce").dropna()
        if len(va) < 2 or len(vb) < 2:
            continue
        _, p  = mannwhitneyu(va, vb, alternative="two-sided")
        stars = p_to_stars(p)
        x1, x2 = x_pos[ca], x_pos[cb]
        y = y_max + off * y_range
        ax.plot([x1, x1, x2, x2], [y, y + 0.02*y_range, y + 0.02*y_range, y],
                color="black", linewidth=BRACKET_LW)
        ax.text((x1+x2)/2, y + 0.03*y_range, stars,
                ha="center", va="bottom",
                fontsize=ANNOT_FONTSIZE, fontweight="bold")


def add_median_lines(ax, df, metric, condition_order=CONDITION_ORDER):
    for i, cond in enumerate(condition_order):
        vals = pd.to_numeric(
            df.loc[df["condition"] == cond, metric], errors="coerce").dropna()
        if vals.empty:
            continue
        med = vals.median()
        ax.plot([i - 0.15, i + 0.15], [med, med],
                color="white", linewidth=2.0, solid_capstyle="round", zorder=5)
        ax.plot([i - 0.15, i + 0.15], [med, med],
                color="black", linewidth=0.7, solid_capstyle="round", zorder=6)


def style_xticklabels(ax, condition_order=CONDITION_ORDER):
    ax.set_xticks(range(len(condition_order)))
    ax.set_xticklabels(condition_order, fontsize=TICK_FONTSIZE)
    for tick, cond in zip(ax.get_xticklabels(), condition_order):
        tick.set_color(FIG_PALETTE[cond])
        tick.set_fontweight("bold")


def finish_ax(ax, title, ylabel, df=None, metric=None, sig=True):
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    style_xticklabels(ax)
    if sig and df is not None and metric is not None:
        add_sig_brackets(ax, df, metric)
    sns.despine(ax=ax)


def save_fig(fig, filename):
    path = save_dir / filename
    fig.savefig(path, dpi=SAVE_DPI, bbox_inches="tight")
    print(f"Saved: {path}")
    plt.show()
    plt.close(fig)


# -----------------------------------------------------------------------------
# ❸  PLOTS 1–3: Ripley's H Mean ± SEM — one per condition
# -----------------------------------------------------------------------------

for condition_name, h_curves, color in [
    ("PFA",  pfa_h_curves,  FIG_PALETTE["PFA"]),
    ("BP",   bp_h_curves,   FIG_PALETTE["BP"]),
    ("Cryo", cryo_h_curves, FIG_PALETTE["Cryo"]),
]:
    if not h_curves:
        print(f"No Ripley curves for {condition_name}, skipping.")
        continue

    h_arr  = np.vstack(h_curves)
    mean_h = np.nanmean(h_arr, axis=0)
    sem_h  = scipy_sem(h_arr, axis=0, nan_policy="omit")

    peak_idx = np.nanargmax(mean_h)
    peak_r   = radii[peak_idx]
    peak_h   = mean_h[peak_idx]

    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
    for hc in h_curves:
        ax.plot(radii, hc, color=INDIVIDUAL_LINE_COLOR,
                linewidth=INDIV_LW, alpha=0.8)
    ax.plot(radii, mean_h, color=color, linewidth=MEAN_LW,
            label=f"Mean H(r)  |  Peak H = {peak_h:.2f}, r = {peak_r:.1f}")
    ax.fill_between(radii, mean_h - sem_h, mean_h + sem_h,
                    color=color, alpha=SEM_ALPHA, label="SEM")
    ax.axhline(0, color="black", linestyle="--", linewidth=REF_LW,
               label="CSR: H(r) = 0")
    ax.set_xlim(0, radii[-1])
    ax.set_title(f"Ripley's H(r) — {condition_name} (Mean ± SEM)",
                 fontsize=TITLE_FONTSIZE)
    ax.set_xlabel("Radius r", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel("Ripley's H(r)", fontsize=LABEL_FONTSIZE)
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
    sns.despine(ax=ax)
    save_fig(fig, f"fig_ripley_mean_sem_{condition_name}.png")


# -----------------------------------------------------------------------------
# ❹  PLOT 4: Pair Correlation Function — all 3 conditions overlaid
# -----------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
ax.axhline(1, color="black", linestyle="--", linewidth=REF_LW, label="CSR")

for condition_name in CONDITION_ORDER:
    curves = pcf_curves.get(condition_name, [])
    if not curves:
        continue
    color  = FIG_PALETTE[condition_name]
    arr    = np.vstack(curves)
    mean_g = np.nanmean(arr, axis=0)
    sem_g  = scipy_sem(arr, axis=0, nan_policy="omit")
    ax.plot(bin_mids_ref, mean_g, color=color, linewidth=MEAN_LW,
            label=condition_name)
    ax.fill_between(bin_mids_ref, mean_g - sem_g, mean_g + sem_g,
                    color=color, alpha=SEM_ALPHA)

ax.set_xlim(0, bin_mids_ref[-1])
ax.set_title("Pair Correlation Function — All Conditions", fontsize=TITLE_FONTSIZE)
ax.set_xlabel("Radius r (nm)", fontsize=LABEL_FONTSIZE)
ax.set_ylabel("Pair correlation g(r)", fontsize=LABEL_FONTSIZE)
ax.tick_params(labelsize=TICK_FONTSIZE)
ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)
sns.despine(ax=ax)
save_fig(fig, "fig_pcf_all_conditions.png")


# -----------------------------------------------------------------------------
# ❺  PLOT 5: Number of clusters per ROI
# -----------------------------------------------------------------------------

plot_df = dbscan_summary_df[dbscan_summary_df["condition"].isin(CONDITION_ORDER)].copy()
plot_df["n_clusters"] = pd.to_numeric(plot_df["n_clusters"], errors="coerce")
plot_df = plot_df.dropna(subset=["n_clusters"])

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
sns.violinplot(data=plot_df, x="condition", y="n_clusters", hue="condition",
               order=CONDITION_ORDER, hue_order=CONDITION_ORDER, palette=FIG_PALETTE,
               inner=None, cut=VIOLIN_CUT, linewidth=BOX_LW,
               dodge=False, legend=False, ax=ax)
add_median_lines(ax, plot_df, "n_clusters")
sns.swarmplot(data=plot_df, x="condition", y="n_clusters",
              order=CONDITION_ORDER, color="black",
              size=SWARM_SIZE, alpha=0.8, ax=ax)
finish_ax(ax, "Number of Clusters per ROI", "Number of clusters",
          df=plot_df, metric="n_clusters")
save_fig(fig, "fig_n_clusters.png")


# -----------------------------------------------------------------------------
# ❻  PLOT 6: Localizations per cluster
# -----------------------------------------------------------------------------

lpc = per_cluster_df[per_cluster_df["condition"].isin(CONDITION_ORDER)].copy()
lpc["size_locs"] = pd.to_numeric(lpc["size_locs"], errors="coerce")
lpc = lpc.dropna(subset=["size_locs"])

lpc_per_roi = (
    lpc.groupby(["condition", "dataset_index", "file_name", "roi_index"])
    ["size_locs"].median().reset_index()
    .rename(columns={"size_locs": "median_locs_per_cluster"})
)

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
sns.violinplot(data=lpc_per_roi, x="condition", y="median_locs_per_cluster",
               hue="condition", order=CONDITION_ORDER, hue_order=CONDITION_ORDER,
               palette=FIG_PALETTE, inner=None, cut=VIOLIN_CUT,
               linewidth=BOX_LW, dodge=False, legend=False, ax=ax)
add_median_lines(ax, lpc_per_roi, "median_locs_per_cluster")
sns.swarmplot(data=lpc_per_roi, x="condition", y="median_locs_per_cluster",
              order=CONDITION_ORDER, color="black",
              size=SWARM_SIZE, alpha=0.8, ax=ax)
finish_ax(ax, "Localizations per Cluster", "Median localizations per cluster",
          df=lpc_per_roi, metric="median_locs_per_cluster")
save_fig(fig, "fig_locs_per_cluster.png")


# -----------------------------------------------------------------------------
# ❼  PLOT 7: Mean cluster diameter
# -----------------------------------------------------------------------------

plot_df2 = dbscan_summary_df[dbscan_summary_df["condition"].isin(CONDITION_ORDER)].copy()
plot_df2["mean_cluster_diameter_nm"] = pd.to_numeric(
    plot_df2["mean_cluster_diameter_nm"], errors="coerce")
plot_df2 = plot_df2.dropna(subset=["mean_cluster_diameter_nm"])

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
sns.violinplot(data=plot_df2, x="condition", y="mean_cluster_diameter_nm",
               hue="condition", order=CONDITION_ORDER, hue_order=CONDITION_ORDER,
               palette=FIG_PALETTE, inner=None, cut=VIOLIN_CUT,
               linewidth=BOX_LW, dodge=False, legend=False, ax=ax)
add_median_lines(ax, plot_df2, "mean_cluster_diameter_nm")
sns.swarmplot(data=plot_df2, x="condition", y="mean_cluster_diameter_nm",
              order=CONDITION_ORDER, color="black",
              size=SWARM_SIZE, alpha=0.8, ax=ax)
finish_ax(ax, "Mean Cluster Diameter", "Mean cluster diameter (nm)",
          df=plot_df2, metric="mean_cluster_diameter_nm")
save_fig(fig, "fig_mean_cluster_diameter.png")


# -----------------------------------------------------------------------------
# ❽  PLOT 8: Mean distance to 3 nearest cluster centroids
# -----------------------------------------------------------------------------

plot_knn = knn_per_roi[knn_per_roi["condition"].isin(CONDITION_ORDER)].copy()
plot_knn["median_mean_knn_dist_nm"] = pd.to_numeric(
    plot_knn["median_mean_knn_dist_nm"], errors="coerce")
plot_knn = plot_knn.dropna(subset=["median_mean_knn_dist_nm"])

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
sns.violinplot(data=plot_knn, x="condition", y="median_mean_knn_dist_nm",
               hue="condition", order=CONDITION_ORDER, hue_order=CONDITION_ORDER,
               palette=FIG_PALETTE, inner=None, cut=VIOLIN_CUT,
               linewidth=BOX_LW, dodge=False, legend=False, ax=ax)
add_median_lines(ax, plot_knn, "median_mean_knn_dist_nm")
sns.swarmplot(data=plot_knn, x="condition", y="median_mean_knn_dist_nm",
              order=CONDITION_ORDER, color="black",
              size=SWARM_SIZE, alpha=0.8, ax=ax)
finish_ax(ax, f"Mean Distance to {K_NEIGHBOURS} Nearest Cluster Centroids",
          f"Median mean k={K_NEIGHBOURS} NN distance (nm)",
          df=plot_knn, metric="median_mean_knn_dist_nm")
save_fig(fig, "fig_knn3_distance.png")

print("\nAll figure plots complete.")